# SLC46A1 engraftment MD, part 2: wt

Taner Karagol and Alper Karagol.

This notebook runs **wt**: the chicken wild type, the deposited sequence of 7BC6 with nothing substituted.

**Runtime > Change runtime type > GPU, then Run all, then close it.** It stops when the wall budget is nearly spent and resumes where it left off, so a second Run all costs about a minute of setup and continues from the last finished chunk.

There are three part 2 notebooks, one per arm, and each runs the 2 replicates one at a time from the `REPLICATE` setting. They are independent and can run at the same time in separate sessions. Open `SLC46A1_MD_3_compare.ipynb` once the arms have reached the same length.

## The question the three arms answer

The construct humanises 20 positions of the chicken transporter onto the human pharmacophore and adds rescuers at 6 more, and the sequence likelihood says the set holds together above the 0.001 threshold. A likelihood is not a free energy and it says nothing about whether the fold survives in a bilayer, which is what these runs measure.

The wild type is the reference. The primaries arm separates what the humanising substitutions do on their own from what the rescuers change, so a difference between the two humanised arms is attributable to the rescuers and a difference from the wild type is attributable to the humanisation.

Chicken 238 and 239 are the positions where the sequence likelihood and the structure prediction disagree: the positions where the sequence likelihood and the structure prediction disagree, measured in data/af3_models_report.json. Across the five AlphaFold3 models per arm their pLDDT ranges are construct chicken 238 83.5 to 83.9, chicken 239 79.1 to 79.7; primaries chicken 238 86.6 to 87.2, chicken 239 83.0 to 83.6; wt chicken 238 89.4 to 89.7, chicken 239 88.2 to 88.5. Whether that drop is a local prediction artefact or a real loss of order is a question the trajectories can speak to, since RMSF at those residues is measured in all three arms.

## What it measures

After the burn-in fraction, per frame:

- the backbone RMSD against the first frame and the radius of gyration,
- the per-residue RMSF, reported over the whole chain, over the 66 pharmacophore positions the model resolves and at the watched positions,
- 6 named distance pairs, every one of them present in all three arms, listed below,
- the box area per lipid, the thermostat temperature and the pressure, which say whether the bilayer settled.

The pairs measured:

- `R156_to_D164_control`: chicken 156 NH1 to chicken 164 OD1, carried over from md/md_plan.json, same residues in every arm
- `E193_to_R384_control`: chicken 193 OE1 to chicken 384 NH1, carried over from md/md_plan.json, same residues in every arm
- `disulfide_72_306`: chicken 72 SG to chicken 306 SG, the one built disulfide, which should not move in any arm
- `ca_407_to_166`: chicken 407 CA to chicken 166 CA, the pocket pair on backbone atoms, comparable across the arms
- `ca_238_to_241`: chicken 238 CA to chicken 241 CA, a watched position to the rescuer nearest it
- `ca_239_to_241`: chicken 239 CA to chicken 241 CA, a watched position to the rescuer nearest it

Pairs in `md/md_plan.json` that no longer exist in every arm are left out, since a distance measured in one arm and absent in another is not a comparison: `E407N_to_N166`, `E407_to_N166_wildtype`.

Every residue is resolved by matching the spec against the built structure. CHARMM-GUI renumbers the chain, so the shift between the deposited author numbering and the built numbering is measured and the only shift that fits every named position is accepted.

## What it refuses to do

A folder holding neither a membrane build nor a solution build, or holding part of one, which catches an upload that did not finish. A set of solution builds, since the transporter is run in a bilayer. Arms built as different kinds. A folder whose residues at the positions the spec names disagree with it, which catches the three CHARMM-GUI uploads going in the wrong order, including a construct folder declared as the primaries arm, whose spec names a subset of the constructs and agrees with it everywhere it names. A topology whose disulfides are not the 1 the spec records, between the same residues in every arm. A topology whose hydrogen masses were repartitioned when the record says they were not, which moves the mass the 0.002 ps timestep was chosen against and is invisible in the coordinates. A build carrying no lipid, since a transporter run in solution answers a different question. Arms whose thermostat, timestep or bilayer differ, or whose counter-ion counts differ by more than their formal charges do. A chunk that wrote fewer frames than it should, or a concatenation shorter than it reports. A missing GPU, refused at the start instead of hours later.


In [ ]:
PROJECT_DIR     = ""
DRIVE_ROOT      = "/content/drive/MyDrive/MIT/All Project Files"
PROJECT_FOLDER  = "SLC46A1 engraftment"
BUILDS_ROOT     = "/content/drive/MyDrive/MDs/SLC46A1_engraftment"
REPLICATE       = 2
NS_PER_ARM      = 100
TOTAL_NS        = 100
NS_PER_CHUNK    = 5
WALL_BUDGET_HOURS = 23.0
TARGET_FRAMES   = 2000
BURN_FRACTION   = 0.2
CACHE_GROMACS   = True
GMX_VER         = "2026.3"
GMX_SIMD        = "AVX2_256"
REF_T_EXPECTED  = 303.15
MAXWARN         = 10

import os, time
T_START = time.time()
PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
MD = os.path.join(PROJECT, "md")
BUILD = "wt"
BUILD_ORDER = ["construct", "primaries", "wt"]
assert BUILD in BUILD_ORDER
if REPLICATE not in [1, 2]:
    raise SystemExit("STOP: REPLICATE is %r and md/md_plan.json plans %d replicates"
                     % (REPLICATE, 2))
TAG = "%s_r%d" % (BUILD, REPLICATE)
print("this notebook runs arm:", BUILD.upper(), "replicate", REPLICATE)
print("project      ", PROJECT)
if NS_PER_ARM > TOTAL_NS:
    raise SystemExit("STOP: NS_PER_ARM is %g and TOTAL_NS is %g. The write interval "
                     "comes from the planned total, so the total cannot be the "
                     "smaller of the two." % (NS_PER_ARM, TOTAL_NS))
print("sampling     ", NS_PER_ARM, "ns now, of a planned", TOTAL_NS, "ns per replicate")
print("wall budget  ", WALL_BUDGET_HOURS, "hours")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, re, subprocess, shutil, datetime, math, collections
import numpy as np

PREFIX = "/usr/local/gromacs"
os.environ["PATH"] = PREFIX + "/bin:" + os.environ["PATH"]
RESULTS = os.path.join(MD, "results")
INPUTS = os.path.join(MD, "inputs")
DATA = os.path.join(PROJECT, "data")
os.makedirs(RESULTS, exist_ok=True)


SPEC_PATH = os.path.join(INPUTS, "build_spec.json")
if not os.path.isfile(SPEC_PATH):
    raise SystemExit("STOP: no %s. It is written by codes/28_md_notebook_prepare.py "
                     "in this project, so the checklist that said what to build and "
                     "the checks here read one record." % SPEC_PATH)
SPEC = json.load(open(SPEC_PATH))
if sorted(BUILD_ORDER) != sorted(SPEC["builds"]):
    raise SystemExit("STOP: the spec lists the arms %s and this notebook expects %s"
                     % (SPEC["builds"], BUILD_ORDER))
EXPECT = SPEC["expect"]
RES_LO, RES_HI = SPEC["residue_range"]
PSET = sorted(SPEC["position_set"])
WATCH = sorted(SPEC["watch"])
PSET_IN = [p for p in PSET if RES_LO <= p <= RES_HI]
PSET_OUT = [p for p in PSET if p not in PSET_IN]
WATCH_IN = [p for p in WATCH if RES_LO <= p <= RES_HI]
SS_EXPECTED = sorted(tuple(x) for x in SPEC["disulfides_expected"])
CHARGE = SPEC["formal_charge"]
NRES = SPEC["n_residues"]
PAIRS = [ { "name": "R156_to_D164_control", "a": [ 156, "NH1" ], "b": [ 164, "OD1" ], "why": "carried over from md/md_plan.json, same residues in every arm" }, { "name": "E193_to_R384_control", "a": [ 193, "OE1" ], "b": [ 384, "NH1" ], "why": "carried over from md/md_plan.json, same residues in every arm" }, { "name": "disulfide_72_306", "a": [ 72, "SG" ], "b": [ 306, "SG" ], "why": "the one built disulfide, which should not move in any arm" }, { "name": "ca_407_to_166", "a": [ 407, "CA" ], "b": [ 166, "CA" ], "why": "the pocket pair on backbone atoms, comparable across the arms" }, { "name": "ca_238_to_241", "a": [ 238, "CA" ], "b": [ 241, "CA" ], "why": "a watched position to the rescuer nearest it" }, { "name": "ca_239_to_241", "a": [ 239, "CA" ], "b": [ 241, "CA" ], "why": "a watched position to the rescuer nearest it" } ]
print("build spec: %s on %s, %d residues %d to %d, arms %s"
      % (SPEC["project"], SPEC["entry"], NRES, RES_LO, RES_HI, ", ".join(BUILD_ORDER)))
print("substitutions per arm: %s"
      % ", ".join("%s %d" % (a, len(SPEC["substitutions"][a])) for a in BUILD_ORDER))
print("watched positions %s, measured on %d distance pairs" % (WATCH, len(PAIRS)))
if PSET_OUT:
    print("position-set positions before the first resolved residue, left out "
          "of every per-position measurement: %s" % PSET_OUT)

REF_T_WANT = SPEC.get("ref_t_K", SPEC["run_budget"].get("temperature_k"))
REF_T_SOURCE = "the build spec"
if REF_T_WANT is None:
    REF_T_WANT = REF_T_EXPECTED
    REF_T_SOURCE = "this notebook; the spec carries no temperature"
REF_T_WANT = float(REF_T_WANT)
print("thermostat: %g K expected, from %s" % (REF_T_WANT, REF_T_SOURCE))

HMR_RECORDED = bool(SPEC["build_settings"].get("hydrogen_mass_repartitioning"))
LIPIDS_PER_LEAFLET = SPEC["build_settings"]["lipids_per_leaflet_target"]
LIPID_RESNAMES = {"POPC", "CHL1", "CHOL", "DPPC", "POPE", "POPS"}
import re as _re


def _names_arm(rel, arm):
    return bool(_re.search(r"(?<![a-z0-9])%s(?![a-z0-9])"
                           % _re.escape(arm), rel))


CG_MARKERS = ("step5_input.pdb", "step3_input.pdb")
CG_MARKER = " or ".join(CG_MARKERS)
CG_REQUIRED = ["topol.top", "index.ndx"]


def find_run_dirs(roots, builds):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if any(m in filenames for m in CG_MARKERS):
                if dirpath not in [d for _, d, _ in found]:
                    found.append((root, dirpath,
                                  os.path.relpath(dirpath, root).lower()))
                dirnames[:] = []
    out = {}
    for b in builds:
        per_root = [(r, [d for rr, d, rel in found if rr == r and _names_arm(rel, b)])
                    for r in roots]
        per_root = [(r, h) for r, h in per_root if h]
        if not per_root:
            raise SystemExit(
                "STOP: no folder holding a %s carries the word %s below any root "
                "searched, %s. Every folder holding a %s: %s."
                % (CG_MARKER, b, roots, CG_MARKER,
                   [d for _, d, _ in found] or "none"))
        root, hits = per_root[0]
        if len(hits) != 1:
            raise SystemExit(
                "STOP: %d folders under %s carry the word %s, %s. One per build is "
                "needed and this is not guessed." % (len(hits), root, b, sorted(hits)))
        out[b] = hits[0]
        for _, other in per_root[1:]:
            for o in other:
                _have = [m for m in CG_MARKERS
                         if os.path.isfile(os.path.join(out[b], m))]
                for f in (_have or list(CG_MARKERS)):
                    pa, po = os.path.join(out[b], f), os.path.join(o, f)
                    a = os.path.getsize(pa) if os.path.isfile(pa) else None
                    c = os.path.getsize(po) if os.path.isfile(po) else None
                    if a != c:
                        raise SystemExit(
                            "STOP: the %s build is in two places and the copies are "
                            "not the same system. %s has a %s of %s bytes and %s has "
                            "%s. Delete whichever is stale; which one runs is not "
                            "guessed." % (b, out[b], f, a, o, c))
                print("  note: %s is also at %s, same size, and that copy is not used"
                      % (b, o))
    return out


RUN_DIR = find_run_dirs([BUILDS_ROOT, MD], BUILD_ORDER)
print("\nCHARMM-GUI folders:")
for b in BUILD_ORDER:
    miss = [f for f in CG_REQUIRED if not os.path.isfile(os.path.join(RUN_DIR[b], f))]
    extra = "" if not miss else "   MISSING %s" % miss
    print("  %-10s %s%s" % (b, RUN_DIR[b], extra))
    if miss:
        raise SystemExit("STOP: %s is missing %s, so it is not a finished CHARMM-GUI "
                         "GROMACS folder. Unpack the download again and upload the "
                         "whole gromacs/ folder, toppar included." % (RUN_DIR[b], miss))
if not os.path.isdir(os.path.join(RUN_DIR[BUILD_ORDER[0]], "toppar")):
    raise SystemExit("STOP: no toppar/ beside %s. topol.top includes its .itp files by "
                     "relative path, so grompp fails without it."
                     % os.path.basename(RUN_DIR[BUILD_ORDER[0]]))

print("\n--- GPU ---")
HAVE_GPU = False
GPU_NAME = None
try:
    _g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], check=False, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if _g.returncode == 0 and (_g.stdout or "").strip():
        HAVE_GPU = True
        GPU_NAME = _g.stdout.strip().split("\n")[0]
        print(GPU_NAME)
        print("Chunk times scale with this card, so record it with the run: the same "
              "system takes several times longer on a T4 than on an A100.")
    else:
        print("nvidia-smi exited %d, so the runtime claims a GPU it cannot use."
              % _g.returncode)
        print((_g.stdout or "").strip()[:400])
except FileNotFoundError:
    print("no nvidia-smi, so this runtime has no GPU.")
    print("That is fine for the salvage and compare notebooks: they only call "
          "trjconv, trjcat, distance and rms, all of which run on the CPU.")
    print("A production run needs one. Runtime > Change runtime type > T4 GPU.")


def sh(cmd, cwd=None, label=""):
    print("\n$ " + cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = (r.stdout or "").split("\n")
    if len(lines) > 60:
        print("\n".join(lines[:25]))
        print("   ... %d lines omitted ..." % (len(lines) - 45))
        print("\n".join(lines[-20:]))
    else:
        print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError("%s FAILED (exit %d). The output above is the reason."
                           % (label or cmd, r.returncode))


def gmx_version():
    try:
        return subprocess.run(["gmx", "--version"], capture_output=True, text=True,
                              timeout=90).stdout
    except Exception:
        return ""


CACHE = os.path.join(PROJECT, "gromacs_cuda_%s.tar.gz" % GMX_VER)
CACHE_STALE = False

if "CUDA" in gmx_version():
    print("\nCUDA gmx already present in this session")
elif CACHE_GROMACS and os.path.isfile(CACHE):
    print("\nrestoring the cached GROMACS from", CACHE)
    try:
        sh("tar xzf '%s' -C /" % CACHE, label="restore cache")
    except Exception as e:
        print("the cache could not be read (%s); another session may be writing it. "
              "Building from source instead." % e)
        shutil.rmtree(PREFIX, ignore_errors=True)
    if "CUDA" not in gmx_version():
        print("the cached build does not run here, rebuilding from source")
        shutil.rmtree(PREFIX, ignore_errors=True)
        CACHE_STALE = True

if "CUDA" not in gmx_version():
    nvcc = subprocess.run("which nvcc", shell=True, capture_output=True,
                          text=True).stdout.strip()
    if not nvcc:
        raise SystemExit("STOP: nvcc not found, so there is no CUDA toolkit to build "
                         "against. Set the runtime to GPU and press Run all again.")
    print("\nbuilding GROMACS %s. 20 to 40 minutes, once per Drive folder." % GMX_VER)
    sh("apt-get -qq update && apt-get -qq install -y cmake build-essential "
       "libfftw3-dev", label="apt")
    if not os.path.isdir("/content/gromacs-%s" % GMX_VER):
        sh("wget -q https://ftp.gromacs.org/gromacs/gromacs-%s.tar.gz && "
           "tar xzf gromacs-%s.tar.gz" % (GMX_VER, GMX_VER), cwd="/content",
           label="download")
    os.makedirs("/content/gromacs-%s/build" % GMX_VER, exist_ok=True)
    sh("cmake .. -DGMX_BUILD_OWN_FFTW=OFF -DGMX_GPU=CUDA -DGMX_SIMD=%s "
       "-DCMAKE_INSTALL_PREFIX=%s -DGMX_DOUBLE=OFF -DBUILD_TESTING=OFF"
       % (GMX_SIMD, PREFIX), cwd="/content/gromacs-%s/build" % GMX_VER, label="cmake")
    sh("make -j$(nproc) && make install", cwd="/content/gromacs-%s/build" % GMX_VER,
       label="make")
    if "CUDA" not in gmx_version():
        raise SystemExit("STOP: the build finished but gmx reports no CUDA support.")
    if CACHE_GROMACS and (CACHE_STALE or not os.path.isfile(CACHE)):
        print("\ncaching the build to Drive so later sessions skip it")
        tmp = "%s.%d.tmp" % (CACHE, os.getpid())
        try:
            sh("tar czf '%s' %s" % (tmp, PREFIX.lstrip("/")), cwd="/", label="cache")
            os.replace(tmp, CACHE)
        except Exception as e:
            print("could not cache the build (%s); this costs the next session a "
                  "rebuild and nothing else." % e)
            if os.path.isfile(tmp):
                os.remove(tmp)
    elif CACHE_GROMACS:
        print("\nanother session already cached a build, leaving it alone")

print("\n" + gmx_version().split("\n")[0])


In [ ]:
AA = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HSD HSE HSP ILE LEU LYS MET PHE PRO "
         "SER THR TRP TYR VAL".split())


def gmx(*args, label="", stdin_text=None, quiet=False, check=True):
    cmd = ["gmx"] + [str(a) for a in args]
    if not quiet:
        print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, text=True, input=stdin_text,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    lines = out.split("\n")
    if not quiet:
        if len(lines) > 80:
            print("\n".join(lines[:30]))
            print("   ... %d lines omitted ..." % (len(lines) - 60))
            print("\n".join(lines[-30:]))
        else:
            print(out)
    if r.returncode != 0 and check:
        if quiet:
            print(out[-3000:])
        raise RuntimeError("%s failed (exit %d). The output above is the reason."
                           % (label or cmd[1], r.returncode))
    return r.returncode, out


def must_exist(path, what):
    if not os.path.isfile(path):
        raise RuntimeError("%s reported success but %s was not written." % (what, path))


def read_pdb_atoms(path):
    out = []
    for L in open(path, errors="replace"):
        if not L.startswith(("ATOM", "HETATM")):
            continue
        seg = L[72:76].strip() or L[21].strip() or "?"
        out.append((int(L[6:11]), seg, int(L[22:26]), L[17:20].strip(),
                    L[12:16].strip()))
    return out


def protein_segments(atoms):
    rn = {}
    for ser, seg, ri, name, an in atoms:
        rn.setdefault((seg, ri), name)
    return sorted({seg for (seg, ri), name in rn.items() if name in AA})


def progress(**kw):
    p = os.path.join(RESULTS, "progress_%s.json" % BUILD)
    cur = json.load(open(p)) if os.path.isfile(p) else {}
    cur.update(kw)
    cur["updated"] = datetime.datetime.now().isoformat(timespec="seconds")
    cur["elapsed_hours"] = round((time.time() - T_START) / 3600.0, 2)
    json.dump(cur, open(p, "w"), indent=2)
    return cur


def budget_left():
    return WALL_BUDGET_HOURS - (time.time() - T_START) / 3600.0


def xtc_frames(path):
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    m = re.search(r"^\s*Coords\s+(\d+)", out, re.M)
    if m:
        return int(m.group(1))
    m = re.search(r"Last frame\s+(\d+)", out)
    if m:
        return int(m.group(1)) + 1
    if r.returncode != 0:
        return None
    raise SystemExit(
        "STOP: gmx check read %s and exited 0, but its output carried neither a "
        "Coords row nor a Last frame line, so the frame count is unknown. Fix "
        "this parse rather than letting every chunk be re-run as unsound. The "
        "first lines of the output were:\n%s"
        % (os.path.basename(path), "\n".join(out.split("\n")[:12])))


def xtc_last_time(path):
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    t = None
    for L in (r.stdout or "").split("\n"):
        m = re.search(r"frame\s+\d+\s+time\s+([-\d.eE+]+)", L)
        if m:
            t = float(m.group(1))
    return t


def want_frames(mdp, ns):
    dt = float(mdp_get(mdp, "dt"))
    every = int(mdp_get(mdp, "nstxout-compressed"))
    return int(round(ns * 1000.0 / dt / every))


def ndx_args(group="SOLU"):
    if not os.path.isfile("index.ndx"):
        raise SystemExit(
            "STOP: no index.ndx in %s, so the group %s cannot be resolved. It comes "
            "from the CHARMM-GUI download and is staged in with the other inputs, so "
            "if it is missing here the staging is wrong."
            % (os.getcwd(), group))
    return ("-n", "index.ndx")


def stage_in(run, work, n_chunks, per_chunk):
    os.makedirs(work, exist_ok=True)
    heavy = ("prod_all.xtc", "prod_nojump.xtc", "prod_fit.xtc")
    for name in sorted(os.listdir(run)):
        src = os.path.join(run, name)
        if os.path.isdir(src):
            if name == "toppar":
                shutil.copytree(src, os.path.join(work, name), dirs_exist_ok=True)
            continue
        if re.match(r"prod_\d\d\.xtc$", name) or name in heavy:
            continue
        shutil.copy2(src, os.path.join(work, name))
    kept = 0
    for i in range(1, n_chunks + 1):
        nm = "prod_%02d" % i
        src = os.path.join(run, nm + ".xtc")
        if not (os.path.isfile(src) and os.path.isfile(os.path.join(work, nm + ".gro"))):
            print("  %s absent, so it and every chunk after it is re-run" % nm)
            break
        shutil.copy2(src, os.path.join(work, nm + ".xtc"))
        got = xtc_frames(os.path.join(work, nm + ".xtc"))
        if got is None or got < per_chunk - 1:
            os.remove(os.path.join(work, nm + ".xtc"))
            print("  %s holds %s of %d frames, so it and every chunk after it is "
                  "re-run" % (nm, got, per_chunk))
            break
        kept += 1
    for i in range(kept + 1, n_chunks + 1):
        for ext in (".xtc", ".gro", ".cpt", "_prev.cpt", ".tpr", ".log", ".edr"):
            f = os.path.join(work, "prod_%02d%s" % (i, ext))
            if os.path.isfile(f):
                os.remove(f)
    print("  staged into %s, %d of %d chunks reusable" % (work, kept, n_chunks))
    return kept


DRIVE_PENDING = [0]
DRIVE_SENT = {}


def stage_out(work, run, names):
    for nm in names:
        src = os.path.join(work, nm)
        if not os.path.isfile(src):
            continue
        dst = os.path.join(run, nm)
        shutil.copy2(src, dst)
        got, want = os.path.getsize(dst), os.path.getsize(src)
        if got != want:
            raise SystemExit("STOP: %s is %d bytes in the Drive mount against %d on "
                             "disk, so even the cached copy is short."
                             % (nm, got, want))
        DRIVE_PENDING[0] += want
        DRIVE_SENT[dst] = want


def drive_wait(why="", floor=20.0, mbps=12.0):
    mb = DRIVE_PENDING[0] / 1e6
    DRIVE_PENDING[0] = 0
    if mb <= 0:
        return
    secs = max(floor, mb / mbps)
    print("  waiting %.0f s for Drive to take %.0f MB%s"
          % (secs, mb, " after " + why if why else ""), flush=True)
    os.sync()
    time.sleep(secs)


def drive_verify():
    try:
        from google.colab import drive as _drive
        os.sync()
        print("\nflushing the Drive mount, then checking what Drive actually holds")
        _drive.flush_and_unmount()
        _drive.mount("/content/drive")
    except Exception as _e:
        print("\ncould not flush and remount Drive (%s), so the uploads are "
              "unverified. Leave this tab open for a couple of minutes before "
              "closing it." % _e)
        return
    bad, n = [], 0
    for path, want in sorted(DRIVE_SENT.items()):
        n += 1
        got = os.path.getsize(path) if os.path.isfile(path) else None
        if got != want:
            bad.append((os.path.basename(path), got, want))
    if bad:
        for nm, got, want in bad:
            print("   %-22s %s bytes on Drive against %d written" % (nm, got, want))
        raise SystemExit(
            "STOP: %d of %d files are short or missing on Drive after the flush, so "
            "the upload did not finish. The frames are not lost: open this notebook "
            "again and press Run all, which re-sends what is short and skips every "
            "chunk that is whole." % (len(bad), n))
    print("Drive holds all %d uploaded files at full size. Safe to close the runtime."
          % n)


print("helpers ready, MAXWARN =", MAXWARN)


In [ ]:
def psf_disulfides(run):
    path = os.path.join(run, naming(run)["sys"] + ".psf")
    if not os.path.isfile(path):
        raise SystemExit("STOP: no %s, so the disulfides cannot be verified" % path)
    lines = open(path, errors="replace").read().splitlines()
    i = next(k for k, l in enumerate(lines) if "!NATOM" in l)
    nat = int(lines[i].split()[0])
    sg = {}
    for l in lines[i + 1:i + 1 + nat]:
        f = l.split()
        if len(f) >= 5 and f[4] == "SG" and f[3] == "CYS":
            sg[int(f[0])] = (f[1], int(f[2]))
    j = next(k for k, l in enumerate(lines) if "!NBOND" in l)
    nb = int(lines[j].split()[0])
    tok = []
    for l in lines[j + 1:]:
        tok += [int(x) for x in l.split()]
        if len(tok) >= 2 * nb:
            break
    return sorted({tuple(sorted((sg[a], sg[b])))
                   for a, b in zip(tok[0::2], tok[1::2]) if a in sg and b in sg})


PROTOMER = {"HSD": "HIS", "HSE": "HIS", "HSP": "HIS", "ASPP": "ASP",
            "GLUP": "GLU", "LSN": "LYS", "CYM": "CYS"}


def canon(name):
    return PROTOMER.get(name, name)


def resolve_numbering(run, arm, PDB):
    res, segs = {}, protein_segments(PDB)
    if len(segs) != 1:
        raise SystemExit("STOP: %s has %d protein segments (%s) and this structure is "
                         "a monomer, so the upload is not the file it should be."
                         % (arm, len(segs), ",".join(segs)))
    seg = segs[0]
    for ser, sg, ri, name, an in PDB:
        if sg == seg:
            res.setdefault(ri, canon(name))
    nums = sorted(res)
    if len(nums) != NRES:
        raise SystemExit("STOP: %s has %d protein residues and the spec records %d "
                         "between %d and %d. The upload is not the built file."
                         % (arm, len(nums), NRES, RES_LO, RES_HI))
    probe = {int(k): v for k, v in EXPECT[arm].items()
             if RES_LO <= int(k) <= RES_HI}
    if not probe:
        raise SystemExit("STOP: the spec names no residue for %s, so the numbering "
                         "cannot be resolved against it" % arm)
    fits = []
    for k in range(min(probe) - nums[-1], min(probe) - nums[0] + 1):
        if all(res.get(p - k) == v for p, v in probe.items()):
            fits.append(k)
    if len(fits) != 1:
        if not fits:
            scored = []
            for q in range(min(probe) - nums[-1], min(probe) - nums[0] + 1):
                miss = [(p, res.get(p - q), v) for p, v in sorted(probe.items())
                        if res.get(p - q) != v]
                scored.append((len(miss), q, miss))
            scored.sort(key=lambda t: (t[0], abs(t[1])))
            n_miss, q_best, miss = scored[0]
            raise SystemExit("STOP: no shift of the residue numbering puts the %d "
                             "residues the spec names where this build has them. The "
                             "closest shift is %d, wrong at %d of %d, first %s, each "
                             "one as position, built residue, spec residue. This is "
                             "not the arm it claims to be, or the three CHARMM-GUI "
                             "uploads went in the wrong order."
                             % (len(probe), q_best, n_miss, len(probe), miss[:4]))
        raise SystemExit("STOP: %d shifts of the residue numbering fit the %d residues "
                         "the spec names, %s, so the mapping is not determined and "
                         "every measured position could be the wrong one."
                         % (len(fits), len(probe), fits))
    k = fits[0]
    missing = [p for p in PSET_IN if (p - k) not in res]
    if missing:
        raise SystemExit("STOP: %d of the %d position-set residues fall outside the "
                         "built chain at the resolved shift, first %s"
                         % (len(missing), len(PSET_IN), missing[:6]))
    matched = []
    for other in BUILD_ORDER:
        o = {int(q): v for q, v in EXPECT[other].items() if RES_LO <= int(q) <= RES_HI}
        if o and all(res.get(q - k) == v for q, v in o.items()):
            matched.append((other, set(o)))
    tighter = [o for o, pos in matched if o != arm and pos > set(probe)]
    if tighter:
        raise SystemExit("STOP: this folder matches the spec for %s as well as for "
                         "%s, and %s names every position %s does and more. The "
                         "upload is the %s build under the wrong name."
                         % (", ".join(tighter), arm, tighter[0], arm, tighter[0]))
    same = [o for o, pos in matched if o != arm and pos == set(probe)]
    if same:
        raise SystemExit("STOP: this folder matches %s and %s at the same positions, "
                         "so the spec does not distinguish the two arms and nothing "
                         "measured here would tell them apart."
                         % (arm, ", ".join(same)))
    print("  identity ok: %s matches the spec at all %d named positions, and no arm "
          "constrains this folder more tightly, shift %d, %d residues, %d atoms"
          % (arm, len(probe), k, len(nums), len(PDB)))
    return k, seg, len(nums)


def naming(run):
    for kind, sysname, prod in (("membrane", "step5_input", "step7_production.mdp"),
                                ("solution", "step3_input", "step5_production.mdp")):
        have = [f for f in (sysname + ".pdb", sysname + ".gro", sysname + ".psf", prod)
                if os.path.isfile(os.path.join(run, f))]
        if len(have) == 4:
            return dict(kind=kind, sys=sysname, prod=prod)
        if have:
            raise SystemExit("STOP: %s looks like a %s build and holds only %s of the "
                             "four files one needs, %s. The CHARMM-GUI download is "
                             "incomplete, or its upload did not finish."
                             % (run, kind, len(have), sorted(have)))
    raise SystemExit("STOP: %s holds neither a membrane build (step5_input with "
                     "step7_production.mdp) nor a solution build (step3_input with "
                     "step5_production.mdp). It should be the contents of the "
                     "gromacs/ folder from inside the CHARMM-GUI download." % run)


def hydrogen_masses(run):
    heavy = []
    itps = sorted(glob.glob(os.path.join(run, "toppar", "*.itp")))
    itps.sort(key=lambda f: 0 if os.path.basename(f).startswith("PRO") else 1)
    for path in itps + [os.path.join(run, "topol.top")]:
        if not os.path.isfile(path):
            continue
        inatoms = False
        for line in open(path, errors="replace"):
            s = line.split(";")[0].strip()
            if s.startswith("["):
                inatoms = s.replace(" ", "").lower().startswith("[atoms]")
                continue
            if not inatoms or not s:
                continue
            f = s.split()
            if len(f) >= 8 and f[4].startswith("H"):
                try:
                    heavy.append(float(f[7]))
                except ValueError:
                    pass
        if heavy:
            break
    return heavy


def equilibrate(run):
    mins = [f for f in ("step6.0_minimization.mdp", "step4.0_minimization.mdp")
            if os.path.isfile(f)]
    if not mins:
        raise SystemExit("STOP: no step6.0 or step4.0 minimisation mdp in %s, so the "
                         "CHARMM-GUI download is incomplete." % run)
    mini = mins[0][:-4]
    if not os.path.isfile(mini + ".tpr"):
        gmx("grompp", "-f", mini + ".mdp", "-o", mini + ".tpr", "-c",
            SYS + ".gro", "-r", SYS + ".gro", "-p", "topol.top",
            "-n", "index.ndx", "-maxwarn", MAXWARN, label="grompp minimisation")
        must_exist(mini + ".tpr", "grompp minimisation")
    if not os.path.isfile(mini + ".gro"):
        gmx("mdrun", "-v", "-deffnm", mini, label="minimisation")
        must_exist(mini + ".gro", "minimisation")
    prev = mini
    stages = sorted(glob.glob("step6.?_equilibration.mdp")) or \
             sorted(glob.glob("step4.?_equilibration.mdp"))
    if not stages:
        raise SystemExit("STOP: no step6.* or step4.* equilibration mdp in %s. A "
                         "CHARMM-GUI build writes at least one, so the download is "
                         "incomplete." % run)
    print("  equilibration stages: %s" % ", ".join(s[:-4] for s in stages))
    for mdp in stages:
        name = mdp[:-4]
        if os.path.isfile(name + ".gro"):
            prev = name
            continue
        gmx("grompp", "-f", mdp, "-o", name + ".tpr", "-c", prev + ".gro", "-r",
            SYS + ".gro", "-p", "topol.top", "-n", "index.ndx",
            "-maxwarn", MAXWARN, label="grompp " + name)
        must_exist(name + ".tpr", "grompp " + name)
        gmx("mdrun", "-v", "-deffnm", name, label=name)
        must_exist(name + ".gro", name)
        prev = name
    return prev

def xvg(path, what):
    must_exist(path, what)
    D = np.loadtxt(path, comments=["#", "@"])
    if D.ndim == 1:
        D = D.reshape(-1, len(D)) if D.size else D.reshape(0, 0)
    if not D.size:
        raise SystemExit("STOP: %s is empty, so %s produced no data" % (path, what))
    return D


def sel_atom(shift, pos, atom):
    return 'group "SOLU" and resid %d and name %s' % (pos - shift, atom)


def pair_distances(tpr, shift):
    sel = [" ".join([sel_atom(shift, p["a"][0], p["a"][1]), "plus",
                     sel_atom(shift, p["b"][0], p["b"][1])]) for p in PAIRS]
    gmx("distance", "-s", tpr, "-f", "prod_fit.xtc", "-n", "index.ndx",
        "-select", ";".join(sel), "-oall", "pairs.xvg", "-xvg", "none",
        label="named distance pairs", quiet=True)
    D = xvg("pairs.xvg", "gmx distance")
    if D.shape[1] - 1 != len(PAIRS):
        raise SystemExit("STOP: gmx distance returned %d columns for %d pairs, so a "
                         "selection matched no atom or more than one and the columns "
                         "no longer line up with the pairs."
                         % (D.shape[1] - 1, len(PAIRS)))
    return D


def measure(run, arm, tpr, shift, seg, nres, natoms):
    def stats(v):
        v = np.asarray(v, dtype=float)
        return dict(mean=float(v.mean()), sd=float(v.std()),
                    p10=float(np.percentile(v, 10)),
                    p90=float(np.percentile(v, 90)),
                    first=float(v[0]), last=float(v[-1]))

    D = pair_distances(tpr, shift)
    n = D.shape[0]
    if n < 10:
        raise SystemExit("STOP: the distance series holds %d frames" % n)
    burn = int(n * BURN_FRACTION)
    t_ps = D[:, 0]
    sampled_ns = float((t_ps[-1] - t_ps[0]) / 1000.0)
    if sampled_ns <= 0:
        raise SystemExit("STOP: the trajectory spans %g ns, so its time axis is wrong"
                         % sampled_ns)
    pairs_out = {}
    for j, p in enumerate(PAIRS):
        v = D[burn:, j + 1] * 10.0
        pairs_out[p["name"]] = dict(stats(v), why=p["why"],
                                    a=p["a"], b=p["b"])

    gmx("rms", "-s", tpr, "-f", "prod_fit.xtc", "-o", "rmsd.xvg", "-xvg", "none",
        "-n", "index.ndx", stdin_text="SOLU\nSOLU\n", label="rmsd", quiet=True)
    rmsd = xvg("rmsd.xvg", "gmx rms")[burn:, 1] * 10.0

    gmx("rmsf", "-s", tpr, "-f", "prod_fit.xtc", "-o", "rmsf.xvg", "-res",
        "-xvg", "none", "-n", "index.ndx", "-b", "%g" % t_ps[burn],
        stdin_text="SOLU\n", label="per-residue rmsf", quiet=True)
    R = xvg("rmsf.xvg", "gmx rmsf")
    rmsf = {int(r[0]): float(r[1]) * 10.0 for r in R}
    if len(rmsf) != nres:
        raise SystemExit("STOP: gmx rmsf returned %d residues and the chain has %d, "
                         "so the per-residue values cannot be mapped back to the "
                         "chicken numbering." % (len(rmsf), nres))
    by_chicken = {p: rmsf[p - shift] for p in PSET_IN if (p - shift) in rmsf}
    if len(by_chicken) != len(PSET_IN):
        raise SystemExit("STOP: %d of the %d resolved position-set residues have "
                         "no rmsf value"
                         % (len(PSET_IN) - len(by_chicken), len(PSET_IN)))

    gmx("gyrate", "-s", tpr, "-f", "prod_fit.xtc", "-o", "gyrate.xvg", "-xvg", "none",
        "-n", "index.ndx", stdin_text="SOLU\n", label="radius of gyration", quiet=True)
    rg = xvg("gyrate.xvg", "gmx gyrate")[burn:, 1] * 10.0

    edrs = sorted(glob.glob("prod_[0-9][0-9].edr"))
    box = {}
    if edrs:
        if not os.path.isfile("prod_all.edr"):
            gmx("eneconv", "-f", *edrs, "-o", "prod_all.edr", label="eneconv",
                quiet=True)
        WANT = ["Box-X", "Box-Y", "Temperature", "Pressure"]
        gmx("energy", "-f", "prod_all.edr", "-o", "energy.xvg",
            stdin_text="\n".join(WANT) + "\n",
            label="box and thermostat", quiet=True)
        legend = []
        for line in open("energy.xvg"):
            if line.startswith("@") and " legend " in line and line.count('"') > 1:
                legend.append(line.split('"')[1])
            elif not line.startswith(("#", "@")):
                break
        E = xvg("energy.xvg", "gmx energy")
        if [w for w in WANT if w not in legend] or len(legend) + 1 != E.shape[1]:
            raise SystemExit("STOP: gmx energy returned the columns %s for the "
                             "terms %s over %d columns, so a column cannot be "
                             "matched to its term and the box, the temperature "
                             "and the pressure would be read from the wrong "
                             "place." % (legend, WANT, E.shape[1]))
        col = dict((name, k + 1) for k, name in enumerate(legend))
        m = E.shape[0] // 5
        bx, by = E[m:, col["Box-X"]] * 10.0, E[m:, col["Box-Y"]] * 10.0
        T = stats(E[m:, col["Temperature"]])
        box = dict(box_x_A=stats(bx), box_y_A=stats(by),
                   area_per_lipid_A2=stats(bx * by / float(LIPIDS_PER_LEAFLET)),
                   temperature_K=T, pressure_bar=stats(E[m:, col["Pressure"]]),
                   columns_read=col)
        if abs(T["mean"] - REF_T_WANT) > 5.0:
            raise SystemExit("STOP: the mean temperature over the sampled part of "
                             "the run is %.2f K and the build runs at %.2f K, so "
                             "either the column mapping is wrong or the thermostat "
                             "is not holding." % (T["mean"], REF_T_WANT))
        if bx.mean() <= 0.0 or by.mean() <= 0.0:
            raise SystemExit("STOP: the box averages %.1f by %.1f Angstrom over "
                             "the sampled part of the run, and a box edge cannot "
                             "be negative." % (bx.mean(), by.mean()))

    out = dict(arm=arm, replicate=int(REPLICATE), tag=TAG, run=run, seed=int(SEED),
               atoms=int(natoms), segment=seg, residues=int(nres),
               numbering_shift=int(shift), frames=int(n), burn=int(burn),
               ns=sampled_ns, ns_requested=float(NS_PER_ARM),
               ns_planned=float(TOTAL_NS), offload_path=offload_path(),
               rmsd_A=stats(rmsd), radius_of_gyration_A=stats(rg),
               pair_distances_A=pairs_out,
               rmsf_A_by_chicken_position=by_chicken,
               rmsf_A_whole_chain=stats(list(rmsf.values())),
               rmsf_A_at_watched=\
                   {str(p): by_chicken[p] for p in WATCH_IN if p in by_chicken},
               watched=WATCH_IN, position_set_size=len(PSET_IN),
               position_set_unresolved=PSET_OUT, box=box,
               ref_t_expected=REF_T_WANT, ref_t_source=REF_T_SOURCE,
               written=datetime.datetime.now().isoformat(timespec="seconds"))
    json.dump(out, open(os.path.join(RESULTS, "%s_report.json" % TAG), "w"), indent=2)
    print("  %d frames over %.1f ns, %d burned in" % (n, sampled_ns, burn))
    print("  backbone RMSD %.2f +/- %.2f A" % (out["rmsd_A"]["mean"],
                                               out["rmsd_A"]["sd"]))
    print("  radius of gyration %.2f +/- %.2f A"
          % (out["radius_of_gyration_A"]["mean"], out["radius_of_gyration_A"]["sd"]))
    print("  RMSF over the chain %.2f A, over the %d position set %.2f A"
          % (out["rmsf_A_whole_chain"]["mean"], len(PSET_IN),
             float(np.mean(list(by_chicken.values())))))
    for p in WATCH_IN:
        if p in by_chicken:
            print("  RMSF at the watched chicken %d %.2f A" % (p, by_chicken[p]))
    for nm in sorted(pairs_out):
        s = pairs_out[nm]
        print("  %-22s %.2f +/- %.2f A" % (nm, s["mean"], s["sd"]))
    if box:
        print("  area per lipid %.1f +/- %.1f A2, thermostat %.2f K"
              % (box["area_per_lipid_A2"]["mean"], box["area_per_lipid_A2"]["sd"],
                 box["temperature_K"]["mean"]))
    return out


CHUNK_HOURS = []


def chunk_cost_hours(build):
    seen = list(CHUNK_HOURS)
    p = os.path.join(RESULTS, "progress_%s.json" % build)
    if os.path.isfile(p):
        try:
            v = json.load(open(p)).get("slowest_chunk_hours")
            if v:
                seen.append(float(v))
        except Exception:
            pass
    return max(seen) if seen else None


GPU_FLAGS = ["-ntmpi", "1", "-nb", "gpu", "-pme", "gpu", "-pmefft", "gpu",
             "-bonded", "gpu", "-update", "gpu"]
OFFLOAD_REFUSED = ("was required, but", "not supported with",
                   "is not supported with", "not implemented with")
OFFLOAD_SEEN = []


def offload_path():
    return ",".join(OFFLOAD_SEEN)


def mdrun_production(name, nsteps, extra):
    rc, out = gmx("mdrun", "-v", "-deffnm", name, "-nsteps", nsteps,
                  *(GPU_FLAGS + list(extra)), label=name, check=False)
    if rc == 0:
        if "gpu-resident" not in OFFLOAD_SEEN:
            OFFLOAD_SEEN.append("gpu-resident")
        return
    if not any(w in out for w in OFFLOAD_REFUSED):
        raise SystemExit("STOP: mdrun failed on %s and the output names no unmet "
                         "offload condition, so the cause is not the offload "
                         "flags. The output above is the reason." % name)
    print("  this runtime refuses the GPU-resident update, so the chunk is rerun "
          "on the default offload path")
    if os.path.isfile(name + ".cpt") and "-cpi" not in extra:
        extra = tuple(extra) + ("-cpi", name + ".cpt")
    gmx("mdrun", "-v", "-deffnm", name, "-nsteps", nsteps, *extra, label=name)
    if "default" not in OFFLOAD_SEEN:
        OFFLOAD_SEEN.append("default")

def produce(run, build, eq_final):
    mdp = PROD_MDP
    if not os.path.isfile(mdp):
        raise SystemExit("STOP: no %s in %s; the CHARMM-GUI download is incomplete"
                         % (mdp, run))
    dt = None
    for L in open(mdp):
        m = re.match(r"\s*dt\s*=\s*([\d.]+)", L)
        if m:
            dt = float(m.group(1))
    if dt is None:
        raise SystemExit("STOP: could not read dt from %s, so the chunk length is "
                         "unknown" % mdp)
    nsteps = int(round(NS_PER_CHUNK * 1000.0 / dt))
    n_chunks = int(math.ceil(float(NS_PER_ARM) / NS_PER_CHUNK))
    per_chunk = want_frames(mdp, NS_PER_CHUNK)
    drive_run = REP_DIR
    prev = eq_final
    for i in range(1, n_chunks + 1):
        name = "prod_%02d" % i
        if os.path.isfile(name + ".gro"):
            got = xtc_frames(name + ".xtc")
            if got is not None and got >= per_chunk - 1:
                prev = name
                continue
        cost = chunk_cost_hours(build)
        need = 0.6 if cost is None else max(0.6, cost * 1.15)
        if budget_left() < need:
            print("  wall budget nearly spent, %.1f h left and a chunk "
                  "costs %s, stopping cleanly after %d of %d chunks"
                  % (budget_left(),
                     "an unmeasured amount" if cost is None
                     else "%.1f h" % cost, i - 1, n_chunks))
            return prev, i - 1, n_chunks
        t0 = time.time()
        resume = os.path.isfile(name + ".cpt") and not os.path.isfile(name + ".gro")
        if not os.path.isfile(name + ".tpr"):
            gmx("grompp", "-f", chunk_mdp(i), "-o", name + ".tpr", "-c",
                prev + ".gro", "-p",
                "topol.top", "-n", "index.ndx", "-maxwarn", MAXWARN,
                label="grompp " + name)
        must_exist(name + ".tpr", "grompp " + name)
        extra = ("-cpi", name + ".cpt") if resume else ()
        if resume:
            print("  %s was interrupted, resuming from its checkpoint" % name)
        mdrun_production(name, nsteps, extra)
        must_exist(name + ".gro", name)
        got = xtc_frames(name + ".xtc")
        if got is None or got < per_chunk - 1:
            raise SystemExit("STOP: %s wrote %s frames and should hold %d. The chunk "
                             "is incomplete, and concatenating it would give a "
                             "trajectory shorter than it reports."
                             % (name, got, per_chunk))
        CHUNK_HOURS.append((time.time() - t0) / 3600.0)
        print("  %s %s done in %.1f min, %d frames"
              % (build, name, (time.time() - t0) / 60.0, got))
        stage_out(os.getcwd(), drive_run,
                  [name + ".gro", name + ".cpt", name + ".xtc"])
        drive_wait(why="chunk %d" % i)
        progress(slowest_chunk_hours=round(max(CHUNK_HOURS), 3),
                 offload=offload_path(),
                 **{build: dict(chunks_done=i, chunks_total=n_chunks)})
        prev = name
    return prev, n_chunks, n_chunks


def concatenate(run, n_chunks, per_chunk):
    chunks = sorted(glob.glob("prod_[0-9][0-9].xtc"))
    if len(chunks) > n_chunks:
        raise SystemExit("STOP: %s holds %d chunk trajectories and the run asks for "
                         "%d: %s" % (os.getcwd(), len(chunks), n_chunks, chunks))
    if not chunks:
        raise SystemExit("STOP: %s holds no chunk trajectory" % os.getcwd())
    have_ns = len(chunks) * NS_PER_CHUNK
    counts = [xtc_frames(c) for c in chunks]
    short = [(c, n) for c, n in zip(chunks, counts)
             if n is None or n < per_chunk - 1]
    if short:
        raise SystemExit("STOP: %s hold fewer than the %d frames a chunk should "
                         "have. Concatenating them would produce a trajectory "
                         "shorter than it claims to be." % (short, per_chunk))
    for f in ("prod_all.xtc", "prod_nojump.xtc", "prod_fit.xtc"):
        if os.path.isfile(f):
            os.remove(f)
    offsets = "".join("%g\n" % (k * NS_PER_CHUNK * 1000.0) for k in range(len(chunks)))
    gmx("trjcat", "-f", *chunks, "-o", "prod_all.xtc", "-settime",
        stdin_text=offsets, label="trjcat")
    must_exist("prod_all.xtc", "trjcat")
    span = xtc_last_time("prod_all.xtc")
    if span is None or span < 0.9 * have_ns * 1000.0:
        raise SystemExit("STOP: prod_all.xtc ends at %s ps and the %d chunks that "
                         "finished should reach about %g ps. The time axis is wrong, "
                         "so every per-frame measurement downstream would be "
                         "mislabelled." % (span, len(chunks), have_ns * 1000.0))
    got = xtc_frames("prod_all.xtc")
    if got is None or got < sum(counts) - len(chunks):
        raise SystemExit("STOP: prod_all.xtc holds %s frames and the %d chunks hold "
                         "%d between them. The concatenation is incomplete."
                         % (got, len(chunks), sum(counts)))
    print("  concatenated %d chunks into %d frames, %d ns"
          % (len(chunks), got, have_ns))
    tpr = sorted(glob.glob("prod_[0-9][0-9].tpr"))[0]
    gmx("trjconv", "-s", tpr, "-f", "prod_all.xtc", "-o", "prod_nojump.xtc",
        *ndx_args(), "-pbc", "nojump", stdin_text="SOLU\n",
        label="trjconv nojump")
    must_exist("prod_nojump.xtc", "trjconv nojump")
    gmx("trjconv", "-s", tpr, "-f", "prod_nojump.xtc", "-o", "prod_fit.xtc",
        *ndx_args(), "-fit", "rot+trans", stdin_text="SOLU\nSOLU\n",
        label="trjconv fit")
    must_exist("prod_fit.xtc", "trjconv fit")
    gmx("trjconv", "-s", tpr, "-f", "prod_all.xtc", "-o", "prod_ref.pdb",
        *ndx_args(), "-dump", "0", stdin_text="SOLU\n", label="reference frame")
    must_exist("prod_ref.pdb", "reference frame")
    return tpr


def mdp_get(path, key):
    for l in open(path, errors="replace"):
        l = l.split(";")[0]
        if "=" in l and l.split("=")[0].strip().lower() == key.lower():
            return l.split("=", 1)[1].strip()
    return None


def mdp_set(path, key, value):
    out, hit = [], False
    for l in open(path, errors="replace"):
        body = l.split(";")[0]
        if "=" in body and body.split("=")[0].strip().lower() == key.lower():
            out.append("%-23s = %s\n" % (key, value)); hit = True
        else:
            out.append(l)
    if not hit:
        out.append("%-23s = %s\n" % (key, value))
    open(path, "w").writelines(out)


def prepare_replicate_dir(run, tag):
    d = os.path.join(run, tag)
    os.makedirs(d, exist_ok=True)
    for name in sorted(os.listdir(run)):
        src = os.path.join(run, name)
        if os.path.isdir(src):
            if name == "toppar":
                shutil.copytree(src, os.path.join(d, name), dirs_exist_ok=True)
            continue
        dst = os.path.join(d, name)
        if not os.path.isfile(dst) or                 (os.path.getsize(dst), int(os.path.getmtime(dst))) !=                 (os.path.getsize(src), int(os.path.getmtime(src))):
            shutil.copy2(src, dst)
    return d


def chunk_mdp(i):
    return FIRST_MDP if i == 1 else CONT_MDP


print("=" * 72)
print("PRE-FLIGHT: the three arms must differ only in their substitutions")
print("=" * 72)
ION = {"POT", "CLA", "SOD", "NA", "CL", "K", "KCL"}
WATER = {"TIP3", "HOH", "SOL", "SPC"}
pre = {}
NAMES = {arm: naming(RUN_DIR[arm]) for arm in BUILD_ORDER}
kinds = {NAMES[a]["kind"] for a in BUILD_ORDER}
if len(kinds) != 1:
    raise SystemExit("STOP: the arms are not all the same kind of CHARMM-GUI build, "
                     "%s. A membrane build and a solution build are not comparable."
                     % {a: NAMES[a]["kind"] for a in BUILD_ORDER})
BUILD_KIND = kinds.pop()
SYS = NAMES[BUILD_ORDER[0]]["sys"]
PROD_MDP = NAMES[BUILD_ORDER[0]]["prod"]
print("  %s build in every arm: system %s, production %s"
      % (BUILD_KIND, SYS, PROD_MDP))
if BUILD_KIND == "solution":
    raise SystemExit("STOP: these are solution builds. The transporter is run in a "
                     "bilayer, so a build with no membrane answers a different "
                     "question.")
for arm in BUILD_ORDER:
    run = RUN_DIR[arm]
    pdb = os.path.join(run, SYS + ".pdb")
    atoms = read_pdb_atoms(pdb)
    res = {}
    for ser, sg, ri, name, an in atoms:
        res.setdefault((sg, ri), name)
    counts = collections.Counter(res.values())
    mdp = os.path.join(run, PROD_MDP)
    hm = hydrogen_masses(run)
    pre[arm] = dict(atoms=len(atoms), prot=len(protein_segments(atoms)),
                    ions={k: v for k, v in counts.items() if k in ION},
                    lipids={k: v for k, v in counts.items() if k in LIPID_RESNAMES},
                    waters=sum(v for k, v in counts.items() if k in WATER),
                    ss=psf_disulfides(run),
                    h_mass=(min(hm), max(hm)) if hm else None,
                    ref_t=mdp_get(mdp, "ref-t") or mdp_get(mdp, "ref_t"),
                    dt=mdp_get(mdp, "dt"))
    print("  %-10s %7d atoms, %d protein segment(s), %d waters, lipids %s, ions %s, "
          "%d disulfide(s), ref_t %s, dt %s"
          % (arm, pre[arm]["atoms"], pre[arm]["prot"], pre[arm]["waters"],
             pre[arm]["lipids"] or "none", pre[arm]["ions"] or "none",
             len(pre[arm]["ss"]), pre[arm]["ref_t"], pre[arm]["dt"]))

for arm in BUILD_ORDER:
    got = sorted(tuple(sorted((a[1], b[1]))) for a, b in pre[arm]["ss"])
    if len(got) != len(SS_EXPECTED):
        raise SystemExit("STOP: %s carries %d disulfide(s) in its topology and the "
                         "build spec records %d, %s. A missing bond loses one the "
                         "protein has, and an extra one holds the fold together by a "
                         "bond it does not."
                         % (arm, len(got), len(SS_EXPECTED), SS_EXPECTED))
    pre[arm]["ss_resid"] = got
if len({tuple(pre[a]["ss_resid"]) for a in BUILD_ORDER}) != 1:
    raise SystemExit("STOP: the arms carry disulfides between different residues, "
                     "%s. No substitution touches a cysteine, so the same pair should "
                     "be built in all three."
                     % {a: pre[a]["ss_resid"] for a in BUILD_ORDER})
    if pre[arm]["prot"] != 1:
        raise SystemExit("STOP: %s has %d protein segments and this structure is a "
                         "monomer" % (arm, pre[arm]["prot"]))
    if not pre[arm]["lipids"]:
        raise SystemExit("STOP: %s carries no lipid, so it was built in solution and "
                         "not in a bilayer. A transporter run without its membrane "
                         "answers a different question." % arm)
    hm = pre[arm]["h_mass"]
    if hm is None:
        raise SystemExit("STOP: no hydrogen mass could be read from %s, so whether "
                         "the masses were repartitioned is unknown" % arm)
    repartitioned = hm[1] > 1.5
    if repartitioned != HMR_RECORDED:
        raise SystemExit("STOP: %s carries hydrogen masses between %.3f and %.3f, "
                         "which is %s, and md/md_plan.json records repartitioning as "
                         "%s. Repartitioning moves the mass the timestep of %s ps was "
                         "chosen against and is invisible in the coordinates, so it "
                         "is a tick box on CHARMM-GUI's force field page that has to "
                         "be the same for every arm. Rebuild, or change the record if "
                         "the plan really has moved."
                         % (arm, hm[0], hm[1],
                            "repartitioned" if repartitioned else "unrepartitioned",
                            "on" if HMR_RECORDED else "off", pre[arm]["dt"]))

for key in ("ref_t", "dt"):
    vals = {pre[a][key] for a in BUILD_ORDER}
    if len(vals) != 1 or None in vals:
        raise SystemExit("STOP: the arms disagree on %s: %s. The comparison needs the "
                         "same conditions on all three."
                         % (key, {a: pre[a][key] for a in BUILD_ORDER}))
TEMPERATURE = pre[BUILD_ORDER[0]]["ref_t"]
got_t = [float(x) for x in str(TEMPERATURE).replace(",", " ").split()]
if not got_t or max(abs(v - REF_T_WANT) for v in got_t) > 0.1:
    raise SystemExit("STOP: CHARMM-GUI built these systems with ref_t %s and this "
                     "line runs at %g K (%s). A run at another temperature is not "
                     "comparable to the rest of the series, and the three arms "
                     "matching each other does not make any of them right. Set the "
                     "temperature on CHARMM-GUI's dynamics page and download again, "
                     "or change temperature_k in md/md_plan.json if the series "
                     "temperature really has moved."
                     % (TEMPERATURE, REF_T_WANT, REF_T_SOURCE))

lip = {a: tuple(sorted(pre[a]["lipids"].items())) for a in BUILD_ORDER}
if len(set(lip.values())) != 1:
    raise SystemExit("STOP: the arms carry different bilayers, %s. The membrane is "
                     "the same in all three by construction, so a difference means "
                     "the three CHARMM-GUI sessions did not use the same lipid "
                     "settings." % {a: pre[a]["lipids"] for a in BUILD_ORDER})

spread_q = max(CHARGE[a] for a in BUILD_ORDER) - min(CHARGE[a] for a in BUILD_ORDER)
for ionname in sorted({k for a in BUILD_ORDER for k in pre[a]["ions"]}):
    got = {a: pre[a]["ions"].get(ionname, 0) for a in BUILD_ORDER}
    if max(got.values()) - min(got.values()) > spread_q:
        raise SystemExit("STOP: the arms carry %s counts %s, a spread of %d, and "
                         "their formal charges %s differ by only %d. A larger gap "
                         "means the three builds were not neutralised at one salt "
                         "concentration."
                         % (ionname, got, max(got.values()) - min(got.values()),
                            {a: CHARGE[a] for a in BUILD_ORDER}, spread_q))
spread_a = max(pre[a]["atoms"] for a in BUILD_ORDER) - \
           min(pre[a]["atoms"] for a in BUILD_ORDER)
if spread_a > 5000:
    raise SystemExit("STOP: the arms differ by %d atoms, which is more than the "
                     "substitutions, their rebuilt side chains and the counter-ions "
                     "can account for, so the boxes differ. Rebuild them at one XY "
                     "size and one water thickness." % spread_a)
print("  %d disulfide(s), the same residues in every arm: %s"
      % (len(SS_EXPECTED), pre[BUILD_ORDER[0]]["ss_resid"]))
print("  hydrogen masses %s in every arm, as recorded"
      % ("repartitioned" if HMR_RECORDED else "unrepartitioned"))
print("  bilayer agrees across arms: %s" % pre[BUILD_ORDER[0]]["lipids"])
print("  thermostat agrees across arms and with the spec: ref_t %s K" % TEMPERATURE)
print("  atom spread %d, within tolerance" % spread_a)
progress(preflight="passed", ref_t=TEMPERATURE, atom_spread=spread_a,
         lipids=pre[BUILD_ORDER[0]]["lipids"], replicate=int(REPLICATE))
print("\npre-flight passed, starting the run\n")


PLANNED_FRAMES = int(SPEC["run_budget"]["frames_per_chunk"]) *                  int(SPEC["run_budget"]["chunks_per_replicate"])
PLANNED_NSTXOUT = int(SPEC["run_budget"]["nstxout_compressed"])
if TARGET_FRAMES != PLANNED_FRAMES:
    raise SystemExit("STOP: TARGET_FRAMES is %d and the spec plans %d frames per "
                     "replicate, %d per chunk over %d chunks. The sampling the record "
                     "states and the sampling this run would write are not the same."
                     % (TARGET_FRAMES, PLANNED_FRAMES,
                        SPEC["run_budget"]["frames_per_chunk"],
                        SPEC["run_budget"]["chunks_per_replicate"]))

for arm in BUILD_ORDER:
    mdp = os.path.join(RUN_DIR[arm], PROD_MDP)
    dt = float(mdp_get(mdp, "dt"))
    want = int(round(TOTAL_NS * 1000.0 / dt / TARGET_FRAMES))
    want = max(100, want - want % 100)
    if want != PLANNED_NSTXOUT:
        raise SystemExit("STOP: the write interval works out at %d steps and the spec "
                         "records %d. The planned length, the timestep and the frame "
                         "count no longer agree with the record."
                         % (want, PLANNED_NSTXOUT))
    have = mdp_get(mdp, "nstxout-compressed")
    if have is None or int(have) != want:
        mdp_set(mdp, "nstxout-compressed", want)
        print("  %-10s nstxout-compressed %s -> %d  (%d frames over the planned %d ns)"
              % (arm, have, want, TOTAL_NS * 1000.0 / dt / want, TOTAL_NS))
    else:
        print("  %-10s nstxout-compressed already %d" % (arm, want))

incomplete = []
for arm in [BUILD]:
    run = RUN_DIR[arm]
    rep = os.path.join(RESULTS, "%s_report.json" % TAG)
    print("\n" + "=" * 72)
    print("ARM %s   REPLICATE %d   %s" % (arm.upper(), REPLICATE, run))
    print("=" * 72)
    if os.path.isfile(rep) and json.load(open(rep)).get("ns", 0) >= 0.95 * NS_PER_ARM:
        print("  already finished at %.1f ns" % json.load(open(rep))["ns"])
        continue
    required = [SYS + ".gro", SYS + ".pdb", SYS + ".psf", "topol.top",
                PROD_MDP, "index.ndx"]
    miss = [f for f in required if not os.path.isfile(os.path.join(run, f))]
    if miss:
        raise SystemExit("STOP: %s is missing %s. It should be the gromacs/ folder "
                         "from inside the CHARMM-GUI download." % (run, miss))
    if not globals().get("HAVE_GPU", False):
        raise SystemExit("STOP: arm %s needs a GPU. Equilibration and production both "
                         "ask mdrun to offload, and section 2 found no usable GPU, so "
                         "this would fail hours from now with a message about offload "
                         "instead of about the runtime. Runtime > Change runtime type "
                         "> GPU, then run from the top." % arm)
    PDB = read_pdb_atoms(os.path.join(run, SYS + ".pdb"))
    with open(os.path.join(run, SYS + ".gro"), errors="replace") as fh:
        fh.readline()
        n_gro = int(fh.readline())
    if len(PDB) != n_gro:
        raise SystemExit("STOP: %s %s.pdb has %d atoms and its .gro has %d. They "
                         "must be the same build." % (arm, SYS, len(PDB), n_gro))
    shift, seg, nres = resolve_numbering(run, arm, PDB)
    ss_chicken = sorted(tuple(sorted((a[1] + shift, b[1] + shift)))
                        for a, b in pre[arm]["ss"])
    if ss_chicken != SS_EXPECTED:
        raise SystemExit("STOP: at the resolved shift the built disulfide(s) sit at "
                         "chicken %s and the build spec records %s. Either the bond "
                         "is between the wrong cysteines or the numbering shift is."
                         % (ss_chicken, SS_EXPECTED))
    print("  disulfide(s) at chicken %s, as the spec records" % ss_chicken)
    n_chunks = int(math.ceil(float(NS_PER_ARM) / NS_PER_CHUNK))
    per_chunk = want_frames(os.path.join(run, PROD_MDP), NS_PER_CHUNK)
    work = os.path.join("/content/work", TAG)
    REP_DIR = prepare_replicate_dir(run, TAG)
    print("  a chunk should hold %d frames" % per_chunk)
    stage_in(REP_DIR, work, n_chunks, per_chunk)
    os.chdir(work)

    SEED = 20261100 + REPLICATE
    FIRST_MDP, CONT_MDP = "prod_first.mdp", "prod_cont.mdp"
    for name, first in ((FIRST_MDP, True), (CONT_MDP, False)):
        shutil.copyfile(PROD_MDP, name)
        mdp_set(name, "gen-vel", "yes" if first else "no")
        mdp_set(name, "continuation", "no" if first else "yes")
        mdp_set(name, "ld-seed", SEED)
        if first:
            mdp_set(name, "gen-temp", "%g" % REF_T_WANT)
            mdp_set(name, "gen-seed", SEED)
    print("  replicate %d seeded with %d" % (REPLICATE, SEED))

    eq = equilibrate(work)
    eq_files = sorted(set(glob.glob("step6.*.gro") + glob.glob("step6.*.cpt")
                          + glob.glob("step4.*.gro") + glob.glob("step4.*.cpt")))
    if not eq_files:
        raise SystemExit("STOP: equilibration left no stage output in %s, so there "
                         "is nothing to carry to Drive and nothing to start "
                         "production from" % work)
    stage_out(work, REP_DIR, eq_files)
    drive_wait(why="equilibration of " + TAG)
    print("  equilibration staged to Drive, %d files" % len(eq_files))
    progress(**{TAG: dict(stage="equilibrated")})
    prev, done, total = produce(work, TAG, eq)
    if done == 0:
        incomplete.append((TAG, 0, total))
        continue
    tpr = concatenate(work, done, per_chunk)
    measure(run, arm, tpr, shift, seg, nres, len(PDB))
    stage_out(work, REP_DIR,
              ["prod_fit.xtc", "prod_ref.pdb", os.path.basename(tpr), "pairs.xvg",
               "rmsd.xvg", "rmsf.xvg", "gyrate.xvg", "energy.xvg",
               FIRST_MDP, CONT_MDP])
    drive_wait(why="the analysis files")
    print("  copied the fitted trajectory and its analysis files to Drive")
    if done < total:
        incomplete.append((TAG, done, total))
    progress(**{TAG: dict(stage="measured", chunks_done=done, chunks_total=total)})

os.chdir("/content")
drive_verify()
if incomplete:
    for b, d, t in incomplete:
        print("%s reached %d of %d chunks" % (b, d, t))
    print("\nThe wall budget ran out before %s reached %d ns. Nothing is lost: the "
          "chunks that finished are measured and reported, and opening this notebook "
          "again and pressing Run all continues from the last one. The cached GROMACS "
          "means the next session starts in about a minute." % (TAG, NS_PER_ARM))
print("\n%s finished. Run the other arms and replicates in their own sessions, then "
      "open SLC46A1_MD_3_compare.ipynb." % TAG.upper())


## Done

The report for this replicate is in `md/results/<arm>_r<n>_report.json` on Drive, with the fitted trajectory, its reference frame and every xvg in the replicate folder beside the CHARMM-GUI folder it came from. Nothing of value is left on the virtual machine.

The report carries the length actually sampled, so a session that ran out of wall budget still produces one.

Run the other arms and replicates in their own sessions, then open `SLC46A1_MD_3_compare.ipynb`.
